## **Coleta de dados das Cooperativas**
Aqui serão carregados cada um dos datasets originais e transportados para dentro de um novo dataset que será a base para o projeto.

Os arquivos foram baixados do repositório e importados manualmente para o ambiente deste Notebook.

In [52]:
#Seção de importação de bibliotecas utilizadas
import pandas as pd
import numpy as np
import re


In [53]:
#################################################################################################
# Rotina de carregamento dos datasets e concatenação com o dataset final
# Este código funciona desde que os arquivos de dados coletados (Dados001.xlsx a Dados012.xlsx)
# Esteja presentes
#################################################################################################
# Lista para armazenar temporariamente os dataframes
lista_dfs = []

# Loop para carregar os arquivos Dados001.xlsx até Dados012.xlsx
for i in range(1, 13):
    nome_arquivo = f"Dados{i:03d}.xlsx"
    df_temp = pd.read_excel(nome_arquivo)
    lista_dfs.append(df_temp)

# Concatenando todos em um único dataframe
dados_credito = pd.concat(lista_dfs, ignore_index=True)

# Mostra as primeiras linhas para confirmação
dados_credito.head()

,Carteira,TipoTomador,ReceitaBrutaMensal,ValorTotaBem,AnoEntrada,AnoContratação,MesContratacao,QuantidadeParcelas,TotalContratacoes,TotalContratacoesAtraso,ValorContratado,PrazoTotal,AtividadeRenda,CodigoIBGE,Alvo
0,5.0,PJ,379742.63,994309.60,2010,2009,6,15.0,10,1,87778.19,15.0,5710831,3133808,Adimplente
1,5.0,PF,12084.19,175789.18,2025,2022,3,7.0,21,2,136605.06,7.0,4133,3133808,Inadimplente
2,5.0,PJ,905038.36,661261.97,2012,2012,3,22.0,70,2,553391.09,22.0,1504510,3133808,Inadimplente
3,5.0,PF,11085.52,204825.81,2025,2015,5,32.0,126,0,790378.76,32.0,1834,3133808,Adimplente
4,5.0,PJ,564043.07,376003.30,2012,2021,1,15.0,77,0,639604.43,15.0,5638394,3133808,Adimplente


In [55]:
dados_credito.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21352 entries, 0 to 21351
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Carteira                 18347 non-null  float64
 1   TipoTomador              21352 non-null  object 
 2   ReceitaBrutaMensal       21352 non-null  float64
 3   ValorTotaBem             21352 non-null  float64
 4   AnoEntrada               21352 non-null  int64  
 5   AnoContratação           21352 non-null  int64  
 6   MesContratacao           21352 non-null  int64  
 7   QuantidadeParcelas       21352 non-null  float64
 8   TotalContratacoes        21352 non-null  int64  
 9   TotalContratacoesAtraso  21352 non-null  int64  
 10  ValorContratado          21352 non-null  float64
 11  PrazoTotal               21352 non-null  float64
 12  AtividadeRenda           21352 non-null  object 
 13  CodigoIBGE               21352 non-null  int64  
 14  Alvo                  

In [56]:
contagem_ibge = dados_credito['CodigoIBGE'].value_counts().reset_index()
contagem_ibge.columns = ['CodigoIBGE', 'quantidade']
contagem_ibge

,CodigoIBGE,quantidade
0,5107909,5000
1,3122306,3150
2,1302603,2100
3,4120606,2000
4,3516200,1565
5,5218805,1500
6,3133808,1150
7,2931806,1100
8,4305108,1032
9,1100320,1000


In [58]:

relatorio = []

for i in range(1, 13):
    nome_arquivo = f"Dados{i:03d}.xlsx"
    df = pd.read_excel(nome_arquivo)
    qtd_linhas = len(df)
    relatorio.append((nome_arquivo, qtd_linhas))

# Gerar texto LaTeX
latex_table = r"""\begin{table}[H]
\centering
\caption{Quantidade de linhas por arquivo}
\begin{tabular}{l r}
\hline
\textbf{Arquivo} & \textbf{Linhas} \\
\hline
"""

for nome, linhas in relatorio:
    latex_table += f"{nome} & {linhas} \\\\\n"

latex_table += r"""\hline
\end{tabular}
\end{table}
"""

print(latex_table)

\begin{table}[H]
\centering
\caption{Quantidade de linhas por arquivo}
\begin{tabular}{l r}
\hline
\textbf{Arquivo} & \textbf{Linhas} \\
\hline
Dados001.xlsx & 1150 \\
Dados002.xlsx & 1500 \\
Dados003.xlsx & 5000 \\
Dados004.xlsx & 1565 \\
Dados005.xlsx & 850 \\
Dados006.xlsx & 3150 \\
Dados007.xlsx & 1000 \\
Dados008.xlsx & 1032 \\
Dados009.xlsx & 2100 \\
Dados010.xlsx & 905 \\
Dados011.xlsx & 1100 \\
Dados012.xlsx & 2000 \\
\hline
\end{tabular}
\end{table}



In [64]:
#################################################################################################
# Convertendo dados da coluna TipoTomador  para ficar mais fácil de manusear (PF=1, PJ=2)
#################################################################################################
dados_credito["TipoTomador"] = dados_credito["TipoTomador"].map({"PF": 1, "PJ": 2})

#################################################################################################
# Convertendo dados da coluna Alvo (Adimplente=0, Inadimplente=1)
#################################################################################################
map_alvo_corrigido = {
    "Adimplente": 0,
    "Adimplemente": 0,   # grafia incorreta tratada
    "Inadimplente": 1
}

dados_credito["Alvo"] = dados_credito["Alvo"].map(map_alvo_corrigido)

#################################################################################################
# A variável AtividadeRenda possui dados de dois domínios diferentes, para converter primeiro
# foi removido os hífens e barras
# ======================================================
def normaliza_codigo(codigo):
    codigo = str(codigo).strip()
    # remove hífens e barras
    codigo = re.sub(r"[-/]", "", codigo)
    # mantém apenas dígitos (garantia adicional)
    codigo = re.sub(r"\D", "", codigo)
    return codigo if codigo != "" else None

dados_credito["AtividadeRenda"] = dados_credito["AtividadeRenda"].apply(normaliza_codigo)

In [65]:
dados_credito[['TipoTomador','AtividadeRenda','Alvo']]

,TipoTomador,AtividadeRenda,Alvo
0,2,5710831,0
1,1,4133,1
2,2,1504510,1
3,1,1834,0
4,2,5638394,0
...,...,...,...
21347,1,521110,0
21348,2,9329899,0
21349,1,513505,0
21350,2,161001,0


In [70]:
pip install ydata-profiling

In [71]:
from ydata_profiling import ProfileReport

profile = ProfileReport(dados_credito)
profile.to_file("relatorio.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 15/15 [00:00<00:00, 21.88it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#############################################################################
# Dataset final depois de dados "ajuntados"
#############################################################################
dados_credito.to_csv('/content/dadoscredito05-04.csv')